In [3]:
!pip install wordfreq

   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
    --------------------------------------- 1.0/56.8 MB 25.4 MB/s eta 0:00:03
   - -------------------------------------- 2.1/56.8 MB 5.6 MB/s eta 0:00:10
   -- ------------------------------------- 3.1/56.8 MB 5.3 MB/s eta 0:00:11
   -- ------------------------------------- 4.2/56.8 MB 5.2 MB/s eta 0:00:11
   -- ------------------------------------- 4.2/56.8 MB 5.2 MB/s eta 0:00:11
   --- ------------------------------------ 5.2/56.8 MB 4.4 MB/s eta 0:00:12
   ---- ----------------------------------- 6.3/56.8 MB 4.4 MB/s eta 0:00:12
   ----- ---------------------------------- 7.3/56.8 MB 4.5 MB/s eta 0:00:11
   ----- ---------------------------------- 7.9/56.8 MB 4.2 MB/s eta 0:00:12
   ------ --------------------------------- 8.9/56.8 MB 4.3 MB/s eta 0:00:12
   ------- -------------------------------- 10.0/56.8 MB 4.4 MB/s eta 0:00:11
   ------- -------------------------------- 10.5/56.8 MB 4.6 MB/s eta 0:00:11
   


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from wordfreq import zipf_frequency 

In [ ]:
rt_df = pd.read_csv("data/SPRT_LogLin_216.csv")

gpt2 = pd.read_csv("data_output/bk21_with_gpt2.csv")
g270 = pd.read_csv("data_output/bk21_with_gemma270m.csv")
g12b = pd.read_csv("data_output/bk21_with_gemma12b.csv")

gpt2_sub = gpt2[['ITEM', 'condition', 'cloze_surprisal', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gpt2_uni_surprisal', 
             'bi_surprisal': 'gpt2_bi_surprisal'}
)
g270_sub = g270[['ITEM', 'condition', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gemma270m_uni_surprisal', 
             'bi_surprisal': 'gemma270m_bi_surprisal'}
)
g12b_sub = g12b[['ITEM', 'condition', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gemma12b_uni_surprisal', 
             'bi_surprisal': 'gemma12b_bi_surprisal'}
)

merged = pd.merge(rt_df, gpt2_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g270_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g12b_sub, on=['ITEM', 'condition'], how='left')

# Control variables
# A. Word Length (Characters)
merged['word_length'] = merged['critical_word'].str.len()

# B. Word Position 
merged['word_position'] = merged['position']

# C. Word Frequency (Zipf frequency: log10 scale, 1 to 8)
merged['word_frequency'] = merged['critical_word'].apply(lambda w: zipf_frequency(str(w), 'en'))

# Log Reading Time
merged['log_rt'] = np.log(merged['SUM_3RT'])

# Squared Surprisal (Polynomial terms)
merged['cloze_surp_sq'] = merged['cloze_surprisal']**2

for model in ['gpt2', 'gemma270m', 'gemma12b']:
    merged[f'{model}_uni_sq'] = merged[f'{model}_uni_surprisal']**2
    merged[f'{model}_bi_sq'] = merged[f'{model}_bi_surprisal']**2

merged = merged.dropna(subset=['gpt2_uni_surprisal', 'SUM_3RT', 'cloze_surprisal'])

merged.to_csv("data_output/master_modeling_data.csv", index=False)

print(f"Merge Complete! Total rows: {len(merged)}")

Merge Complete! Total rows: 46092


## Using NLTK 'corpus/words' dictionary instead

In [4]:
rt_df = pd.read_csv("data/SPRT_LogLin_216.csv")

gpt2 = pd.read_csv("data_output/bk21_with_gpt2_nltk.csv")
g270 = pd.read_csv("data_output/bk21_with_gemma270m_nltk.csv")
g12b = pd.read_csv("data_output/bk21_with_gemma12b_nltk.csv")

gpt2_sub = gpt2[['ITEM', 'condition', 'cloze_surprisal', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gpt2_uni_surprisal', 
             'bi_surprisal': 'gpt2_bi_surprisal'}
)
g270_sub = g270[['ITEM', 'condition', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gemma270m_uni_surprisal', 
             'bi_surprisal': 'gemma270m_bi_surprisal'}
)
g12b_sub = g12b[['ITEM', 'condition', 'uni_surprisal', 'bi_surprisal']].rename(
    columns={'uni_surprisal': 'gemma12b_uni_surprisal', 
             'bi_surprisal': 'gemma12b_bi_surprisal'}
)

merged = pd.merge(rt_df, gpt2_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g270_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g12b_sub, on=['ITEM', 'condition'], how='left')

# Control variables
# A. Word Length (Characters)
merged['word_length'] = merged['critical_word'].str.len()

# B. Word Position 
merged['word_position'] = merged['position']

# C. Word Frequency (Zipf frequency: log10 scale, 1 to 8)
merged['word_frequency'] = merged['critical_word'].apply(lambda w: zipf_frequency(str(w), 'en'))

# Log Reading Time
merged['log_rt'] = np.log(merged['SUM_3RT'])

# Squared Surprisal (Polynomial terms)
merged['cloze_surp_sq'] = merged['cloze_surprisal']**2

for model in ['gpt2', 'gemma270m', 'gemma12b']:
    merged[f'{model}_uni_sq'] = merged[f'{model}_uni_surprisal']**2
    merged[f'{model}_bi_sq'] = merged[f'{model}_bi_surprisal']**2

merged = merged.dropna(subset=['gpt2_uni_surprisal', 'SUM_3RT', 'cloze_surprisal'])

merged.to_csv("data_output/master_modeling_data_nltk.csv", index=False)

print(f"Merge Complete! Total rows: {len(merged)}")

Merge Complete! Total rows: 46092


In [9]:
!pip install nltk

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   --------------------------------- ------ 1.3/1.6 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.6 MB/s  0:00:01



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from transformers import AutoTokenizer
import nltk
from nltk.corpus import words

# Download once if needed
try:
    nltk.data.find("corpora/words")
except LookupError:
    nltk.download("words")

nltk_set = set(w.lower() for w in words.words())

models = {
    "gpt2": "gpt2",
    "gemma270m": "google/gemma-3-270m",
    "gemma12b": "google/gemma-3-12b-pt",
}

for short_name, model_id in models.items():
    print(f"\nProcessing {short_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    rows = []
    vocab_size = tokenizer.vocab_size

    for tok_id in range(vocab_size):
        try:
            tok_str = tokenizer.decode([tok_id])
            decoded_clean = tok_str.strip().lower()

            if decoded_clean in nltk_set and decoded_clean.isalpha():
                rows.append({
                    "token_id": tok_id,
                    "token_str": tok_str,
                    "decoded_clean": decoded_clean,
                    "is_alpha": decoded_clean.isalpha()
                })
        except Exception:
            continue

    df = pd.DataFrame(rows)
    out_path = f"data_output/{short_name}_nltk_filtered_vocab.csv"
    df.to_csv(out_path, index=False)

    print(f"Saved: {out_path}")
    print(f"Filtered LLM vocabulary from {vocab_size} down to {len(df)} valid English words.")

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\words.zip.



Processing gpt2...
Saved: data_output/gpt2_nltk_filtered_vocab.csv
Filtered LLM vocabulary from 50257 down to 25950 valid English words.

Processing gemma270m...
Saved: data_output/gemma270m_nltk_filtered_vocab.csv
Filtered LLM vocabulary from 262144 down to 52611 valid English words.

Processing gemma12b...
Saved: data_output/gemma12b_nltk_filtered_vocab.csv
Filtered LLM vocabulary from 262144 down to 52611 valid English words.


In [11]:
import pandas as pd
df = pd.read_csv("data_output/bk21_stimuli_final.csv")
df.head(10).to_csv("data_output/bk21_stimuli_10.csv", index=False)